# Partition and visualize a four-tile COPC area of interest

This example opens the included AHN STAC GeoParquet catalog as one metadata-only `CopcDataset`. It defines an 800 m by 800 m area of interest (AoI) centered on the common corner `(126000, 486000)` of four COPC files, partitions the AoI into six non-overlapping selections, executes those selections with Dask, and renders their geographic arrangement with PyVista.

The complete AoI was measured at 7,489,287 points and 398,804,571 bytes (398.805 decimal MB, 380.33 MiB) when all 18 point dimensions were read without provenance columns. Peak process memory can be higher because the DuckDB PDAL extension currently materializes one source-level point view before exposing Arrow batches.

## Imports and repository paths

Run this notebook from either the repository root or the `examples` directory using the `copc` Conda environment.

In [ ]:
from __future__ import annotations

import os
import zlib
from collections import Counter
from itertools import combinations
from pathlib import Path

import duckdb
import numpy as np
import pyvista as pv
from distributed import Client, LocalCluster
from IPython.display import Image, display
from matplotlib.colors import ListedColormap

from copc_dataset import DaskCopcExecutor, DuckDBCopcReader, open_copc_dataset

cwd = Path.cwd().resolve()
if (cwd / "ahn5.parquet").exists():
    REPOSITORY_ROOT = cwd
elif (cwd.parent / "ahn5.parquet").exists():
    REPOSITORY_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Run the notebook from the repository root or examples directory"
    )

CATALOG_PATH = REPOSITORY_ROOT / "ahn5.parquet"
IMAGE_PATH = REPOSITORY_ROOT / "examples" / "six_selection_classification_grid.png"
CATALOG_PATH

## Open the logical COPC dataset

Opening a compatible STAC GeoParquet catalog uses its source metadata and does not contact the remote COPC assets.

In [ ]:
dataset = open_copc_dataset(CATALOG_PATH)
print(dataset)

dataset_properties = {
    "source_count": dataset.source_count,
    "point_count": dataset.point_count,
    "bounds": dataset.bounds,
    "dimensions": dataset.dimensions,
    "crs_name": dataset.crs.name,
    "crs_authority": dataset.crs.to_authority(),
    "attributes": dict(dataset.attrs),
}
display(dataset_properties)
display(dataset.schema)

In [ ]:
source_columns = [
    "id",
    "point_count",
    "min_x",
    "min_y",
    "min_z",
    "max_x",
    "max_y",
    "max_z",
    "href",
]
display(dataset.sources.select(source_columns))

## Define the four-tile AoI

The AoI extends from 125600 m to 126400 m in X and from 485600 m to 486400 m in Y. Its center is the common corner of four one-kilometer AHN tiles. Omitting Z resolves it to the complete dataset Z range.

In [ ]:
AOI_CENTER = (126_000.0, 486_000.0)
AOI_WIDTH = 800.0
AOI_HEIGHT = 800.0
AOI_X = (AOI_CENTER[0] - AOI_WIDTH / 2, AOI_CENTER[0] + AOI_WIDTH / 2)
AOI_Y = (AOI_CENTER[1] - AOI_HEIGHT / 2, AOI_CENTER[1] + AOI_HEIGHT / 2)

complete_aoi = dataset.selection("four-tile-aoi", x=AOI_X, y=AOI_Y, z=None)
candidate_sources = dataset.candidate_sources(complete_aoi)

print(complete_aoi)
print("Candidate COPCs:")
for source in candidate_sources:
    print(f"  {source.id}: {source.bounds}")
assert len(candidate_sources) == 4

## Partition the AoI into six selections

`PointCloudSelection` uses closed bounds. Directly reusing an internal edge as both an upper and lower bound would therefore duplicate points on that edge. For every non-terminal cell, the upper edge below uses `numpy.nextafter(edge, -inf)`. The next cell starts at the nominal edge, assigning every representable coordinate to exactly one selection.

The nominal grid has three columns and two rows. Each cell is approximately 266.667 m wide and exactly 400 m high.

In [ ]:
GRID_COLUMNS = 3
GRID_ROWS = 2
X_NAMES = ("west", "center", "east")
Y_NAMES = ("south", "north")
VISUALIZATION_COLUMNS = (
    "X",
    "Y",
    "Z",
    "Classification",
    "GpsTime",
    "PointSourceId",
    "ReturnNumber",
)

x_edges = np.linspace(AOI_X[0], AOI_X[1], GRID_COLUMNS + 1)
y_edges = np.linspace(AOI_Y[0], AOI_Y[1], GRID_ROWS + 1)

selections = []
selection_positions = {}
for row in range(GRID_ROWS):
    for column in range(GRID_COLUMNS):
        selection_id = f"{Y_NAMES[row]}-{X_NAMES[column]}"
        max_x = (
            x_edges[column + 1]
            if column == GRID_COLUMNS - 1
            else np.nextafter(x_edges[column + 1], -np.inf)
        )
        max_y = (
            y_edges[row + 1]
            if row == GRID_ROWS - 1
            else np.nextafter(y_edges[row + 1], -np.inf)
        )
        selection = dataset.selection(
            selection_id,
            x=(x_edges[column], max_x),
            y=(y_edges[row], max_y),
            z=None,
            columns=VISUALIZATION_COLUMNS,
        )
        selections.append(selection)
        selection_positions[selection_id] = (row, column)

selections = tuple(selections)
assert len(selections) == GRID_COLUMNS * GRID_ROWS
assert all(
    not first.bounds.intersects(second.bounds)
    for first, second in combinations(selections, 2)
)
assert np.isclose(
    (AOI_WIDTH / GRID_COLUMNS) * (AOI_HEIGHT / GRID_ROWS) * len(selections),
    AOI_WIDTH * AOI_HEIGHT,
)

selection_manifest = [
    {
        "id": selection.id,
        "row_south_to_north": selection_positions[selection.id][0],
        "column_west_to_east": selection_positions[selection.id][1],
        "x_min": selection.bounds.min_x,
        "x_max": selection.bounds.max_x,
        "y_min": selection.bounds.min_y,
        "y_max": selection.bounds.max_y,
        "candidate_sources": len(dataset.candidate_sources(selection)),
    }
    for selection in selections
]
display(selection_manifest)

## Optional all-dimension size verification

Set `COPC_RUN_SIZE_CHECK=1` before starting Jupyter to repeat the remote read that measured the complete all-dimension Arrow payload. The stream is consumed batch by batch and is not retained. This is separate from the visualization read below.

In [ ]:
EXPECTED_AOI_POINTS = 7_489_287
EXPECTED_ARROW_BYTES = 398_804_571
ARROW_BYTE_LIMIT = 400_000_000
RUN_SIZE_VERIFICATION = os.environ.get("COPC_RUN_SIZE_CHECK") == "1"

if RUN_SIZE_VERIFICATION:
    rows = 0
    arrow_bytes = 0
    with DuckDBCopcReader(requests=4).scan(dataset, complete_aoi) as batches:
        for batch in batches:
            rows += batch.num_rows
            arrow_bytes += batch.nbytes
    print(f"Points: {rows:,}")
    print(f"Arrow payload: {arrow_bytes / 1_000_000:.3f} MB")
    print(f"Arrow payload: {arrow_bytes / 1024**2:.2f} MiB")
    assert rows == EXPECTED_AOI_POINTS
    assert arrow_bytes == EXPECTED_ARROW_BYTES
    assert arrow_bytes <= ARROW_BYTE_LIMIT
else:
    print("Skipped. Set COPC_RUN_SIZE_CHECK=1 to repeat the remote size check.")

## Read the six selections with Dask

Every worker counts all points in its selection while retaining only the 200,000 points with the smallest content-derived pseudorandom priorities. Stable point attributes determine the priorities, so batching or traversal order does not change the sample. This is a bounded-memory uniform sample. Only XYZ, classification, counts, and class summaries return to the Dask client.

Two worker processes and two PDAL HTTP request threads per worker keep network and source-level memory concurrency bounded.

In [ ]:
MAX_SAMPLE_POINTS = 200_000


def install_pdal_extension() -> None:
    """Install the DuckDB PDAL extension once before workers load it."""
    connection = duckdb.connect(database=":memory:")
    try:
        connection.execute("INSTALL pdal FROM community")
    finally:
        connection.close()


def stable_point_priorities(batch, selection_id):
    """Return deterministic pseudorandom priorities from point attributes."""
    seed = np.uint64(zlib.crc32(selection_id.encode("utf-8")))
    priorities = np.full(
        batch.num_rows,
        np.uint64(1_469_598_103_934_665_603) ^ seed,
        dtype=np.uint64,
    )
    fnv_prime = np.uint64(1_099_511_628_211)
    for name in ("X", "Y", "Z", "GpsTime"):
        values = np.ascontiguousarray(
            batch[name].to_numpy(zero_copy_only=False), dtype=np.float64
        ).view(np.uint64)
        priorities ^= values
        priorities *= fnv_prime
    for name in ("Classification", "PointSourceId", "ReturnNumber"):
        values = np.asarray(batch[name].to_numpy(zero_copy_only=False), dtype=np.uint64)
        priorities ^= values
        priorities *= fnv_prime

    priorities ^= priorities >> np.uint64(30)
    priorities *= np.uint64(13_765_031_520_994_252_897)
    priorities ^= priorities >> np.uint64(27)
    priorities *= np.uint64(10_723_178_051_746_563_781)
    priorities ^= priorities >> np.uint64(31)
    return priorities


def sample_selection(
    batches, selection, dataset_metadata, max_points=MAX_SAMPLE_POINTS
):
    """Count one selection and return a deterministic uniform point sample."""
    del dataset_metadata
    priorities = np.empty(0, dtype=np.uint64)
    sampled_points = np.empty((0, 3), dtype=np.float64)
    sampled_classes = np.empty(0, dtype=np.uint8)
    class_counts = Counter()
    point_count = 0

    for batch in batches:
        batch_size = batch.num_rows
        point_count += batch_size
        batch_classes = batch["Classification"].to_numpy(zero_copy_only=False)
        class_values, counts = np.unique(batch_classes, return_counts=True)
        class_counts.update(
            {int(value): int(count) for value, count in zip(class_values, counts)}
        )
        batch_points = np.column_stack(
            [batch[name].to_numpy(zero_copy_only=False) for name in ("X", "Y", "Z")]
        )
        candidate_priorities = np.concatenate(
            (priorities, stable_point_priorities(batch, selection.id))
        )
        candidate_points = np.concatenate((sampled_points, batch_points), axis=0)
        candidate_classes = np.concatenate((sampled_classes, batch_classes))

        if candidate_priorities.size > max_points:
            keep = np.argpartition(candidate_priorities, max_points - 1)[:max_points]
            priorities = candidate_priorities[keep]
            sampled_points = candidate_points[keep]
            sampled_classes = candidate_classes[keep]
        else:
            priorities = candidate_priorities
            sampled_points = candidate_points
            sampled_classes = candidate_classes

    return {
        "selection_id": selection.id,
        "point_count": point_count,
        "sample_count": sampled_points.shape[0],
        "points": sampled_points,
        "classification": sampled_classes,
        "class_counts": dict(sorted(class_counts.items())),
    }

In [ ]:
install_pdal_extension()

cluster = LocalCluster(
    n_workers=2,
    threads_per_worker=1,
    processes=True,
    dashboard_address=None,
)
client = Client(cluster)
try:
    executor = DaskCopcExecutor(
        client,
        reader_options={
            "requests": 2,
            "batch_size": 65_536,
            "install_extension": False,
        },
    )
    futures = executor.submit(dataset, selections, processor=sample_selection)
    selection_results = {
        selection_id: future.result() for selection_id, future in futures.items()
    }
finally:
    client.close()
    cluster.close()

partition_point_count = sum(
    result["point_count"] for result in selection_results.values()
)
assert partition_point_count == EXPECTED_AOI_POINTS

result_summary = [
    {
        "selection_id": selection_id,
        "point_count": result["point_count"],
        "sample_count": result["sample_count"],
        "class_counts": result["class_counts"],
    }
    for selection_id, result in selection_results.items()
]
display(result_summary)
print(f"Combined partition count: {partition_point_count:,}")

## Render the geographic selection grid

The first PyVista row is north and the second is south; columns run west to east. All panels use the same top-down camera scale and categorical class colors. Coordinates are translated by the AoI center before rendering to improve display precision, while labels retain RD New coordinates.

In [ ]:
CLASSIFICATION_STYLE = {
    1: ("Unclassified", "#9e9e9e"),
    2: ("Ground", "#8c6d31"),
    6: ("Building", "#d73027"),
    9: ("Water", "#3288bd"),
    26: ("Class 26", "#984ea3"),
}
observed_classes = sorted(
    {
        class_value
        for result in selection_results.values()
        for class_value in result["class_counts"]
    }
)
class_to_index = {value: index for index, value in enumerate(observed_classes)}
class_colors = [
    CLASSIFICATION_STYLE.get(value, (f"Class {value}", "#000000"))[1]
    for value in observed_classes
]
class_annotations = {
    index: f"{value}: {CLASSIFICATION_STYLE.get(value, (f'Class {value}', ''))[0]}"
    for value, index in class_to_index.items()
}
classification_cmap = ListedColormap(class_colors)

plotter = pv.Plotter(
    shape=(GRID_ROWS, GRID_COLUMNS),
    window_size=(1800, 1100),
    border=True,
    off_screen=True,
)
selection_by_id = {selection.id: selection for selection in selections}

for selection_id, result in selection_results.items():
    south_to_north_row, column = selection_positions[selection_id]
    display_row = GRID_ROWS - 1 - south_to_north_row
    plotter.subplot(display_row, column)

    points = result["points"].copy()
    points[:, 0] -= AOI_CENTER[0]
    points[:, 1] -= AOI_CENTER[1]
    cloud = pv.PolyData(points)
    cloud["Classification"] = np.array(
        [class_to_index[int(value)] for value in result["classification"]],
        dtype=np.uint8,
    )

    plotter.add_points(
        cloud,
        scalars="Classification",
        cmap=classification_cmap,
        clim=(-0.5, len(observed_classes) - 0.5),
        n_colors=len(observed_classes),
        annotations=class_annotations,
        point_size=2,
        render_points_as_spheres=False,
        show_scalar_bar=False,
    )
    if display_row == 0 and column == GRID_COLUMNS - 1:
        plotter.add_legend(
            labels=[
                (class_annotations[index], class_colors[index])
                for index in range(len(observed_classes))
            ],
            bcolor="white",
            border=True,
            size=(0.24, 0.24),
            loc="lower right",
            face="rectangle",
        )

    selection = selection_by_id[selection_id]
    center_x = (selection.bounds.min_x + selection.bounds.max_x) / 2 - AOI_CENTER[0]
    center_y = (selection.bounds.min_y + selection.bounds.max_y) / 2 - AOI_CENTER[1]
    center_z = (selection.bounds.min_z + selection.bounds.max_z) / 2
    plotter.view_xy()
    plotter.camera.focal_point = (center_x, center_y, center_z)
    plotter.camera.position = (center_x, center_y, selection.bounds.max_z + 1000)
    plotter.camera.up = (0, 1, 0)
    plotter.camera.parallel_projection = True
    plotter.camera.parallel_scale = AOI_HEIGHT / GRID_ROWS / 2 * 1.05
    plotter.add_text(
        f"{selection_id}\n{result['point_count']:,} points",
        position="upper_left",
        font_size=10,
    )

plotter.show(screenshot=str(IMAGE_PATH), auto_close=True)
display(Image(filename=str(IMAGE_PATH)))
print(f"Saved {IMAGE_PATH}")

## Result interpretation

The six point counts sum to the complete AoI count, demonstrating that the floating-point boundary ownership rule neither duplicates nor loses an AHN point. Sampling affects only rendering: every point is still counted. The separate panels retain their north/south and west/east placement, and the shared classification mapping makes colors directly comparable across selection boundaries.